In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
# Cell 2 — Load Pima Indians Diabetes Dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

df = pd.read_csv(url, names=cols)

print(f"Dataset shape:     {df.shape}")
print(f"Diabetes rate:     {df['Outcome'].mean()*100:.1f}%")
print(f"Missing values:    {df.isnull().sum().sum()}")
print(f"\nFeature stats:")
print(df.describe().round(2))

Dataset shape:     (768, 9)
Diabetes rate:     34.9%
Missing values:    0

Feature stats:
       Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin     BMI  \
count       768.00   768.00         768.00         768.00   768.00  768.00   
mean          3.85   120.89          69.11          20.54    79.80   31.99   
std           3.37    31.97          19.36          15.95   115.24    7.88   
min           0.00     0.00           0.00           0.00     0.00    0.00   
25%           1.00    99.00          62.00           0.00     0.00   27.30   
50%           3.00   117.00          72.00          23.00    30.50   32.00   
75%           6.00   140.25          80.00          32.00   127.25   36.60   
max          17.00   199.00         122.00          99.00   846.00   67.10   

       DiabetesPedigreeFunction     Age  Outcome  
count                    768.00  768.00   768.00  
mean                       0.47   33.24     0.35  
std                        0.33   11.76     0.48  
min

In [3]:
# Cell 3 — Data Cleaning & Feature Engineering
# Zero values in medical features are biologically impossible — treat as missing
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Engineer new features
df_clean['BMI_Age']          = df_clean['BMI'] * df_clean['Age']
df_clean['Glucose_Insulin']  = df_clean['Glucose'] * df_clean['Insulin']
df_clean['High_Glucose']     = (df_clean['Glucose'] > 140).astype(int)
df_clean['Obese']            = (df_clean['BMI'] > 30).astype(int)
df_clean['High_Risk_Age']    = (df_clean['Age'] > 45).astype(int)

X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples:  {X_train.shape[0]}")
print(f"Test samples:      {X_test.shape[0]}")
print(f"Features:          {X_train.shape[1]}")
print(f"\nEngineered features added:")
print(f"  BMI_Age, Glucose_Insulin, High_Glucose, Obese, High_Risk_Age")

Training samples:  614
Test samples:      154
Features:          13

Engineered features added:
  BMI_Age, Glucose_Insulin, High_Glucose, Obese, High_Risk_Age


In [4]:
# Cell 4 — Train XGBoost + Random Forest
# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42, eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=6,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

print(f"=== Model Performance ===")
print(f"XGBoost ROC-AUC:       {xgb_auc:.4f}")
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_model.predict(X_test)))

=== Model Performance ===
XGBoost ROC-AUC:       0.8220
Random Forest ROC-AUC: 0.8102

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.81      0.81       100
           1       0.64      0.63      0.64        54

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154



In [6]:
# Cell 5 — Feature Importance (using Random Forest  for interpretability)
import matplotlib.pyplot as plt
import os

output_dir = r'C:\Users\Gurveer\ds-portfolio\project-23-diabetes-risk-predictor\outputs'
os.makedirs(output_dir, exist_ok=True)

# Random Forest feature importance
feature_names = X_train.columns.tolist()
importances   = rf_model.feature_importances_

importance_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("=== Feature Importance (Random Forest) ===")
print(importance_df.to_string(index=False))

# Plot
fig = px.bar(
    importance_df, x='importance', y='feature',
    orientation='h',
    title='Feature Importance — Diabetes Risk Model',
    template='plotly_dark',
    color='importance',
    color_continuous_scale='Reds'
)
fig.update_layout(
    width=800, height=500,
    yaxis=dict(autorange='reversed')
)
fig.show()

importance_df.to_csv(f'{output_dir}\\feature_importance.csv', index=False)
print("\nFeature importance saved")

=== Feature Importance (Random Forest) ===
                 feature  importance
                 Glucose    0.215635
         Glucose_Insulin    0.154304
                 BMI_Age    0.129857
                     BMI    0.094539
            High_Glucose    0.081470
                     Age    0.073919
DiabetesPedigreeFunction    0.070059
                 Insulin    0.044053
             Pregnancies    0.040337
           SkinThickness    0.038586
           BloodPressure    0.034751
                   Obese    0.018775
           High_Risk_Age    0.003716



Feature importance saved


In [7]:
# Cell 6 — Risk Score Dashboard & Lifestyle Interventions
import os

# Assign risk tiers
xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
df_test     = X_test.copy()
df_test['risk_score'] = xgb_scores
df_test['risk_tier']  = pd.cut(
    df_test['risk_score'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Moderate Risk', 'High Risk']
)
df_test['actual'] = y_test.values

# Risk distribution plot
fig = px.histogram(
    df_test, x='risk_score', color='risk_tier',
    nbins=40,
    title='Diabetes Risk Score Distribution',
    labels={'risk_score': 'Predicted Diabetes Probability'},
    template='plotly_dark',
    color_discrete_map={
        'Low Risk':      'seagreen',
        'Moderate Risk': 'orange',
        'High Risk':     'crimson'
    }
)
fig.update_layout(width=800, height=450)
fig.show()

# Intervention recommendations
interventions = {
    'High Risk': [
        "Consult a physician immediately for glucose tolerance test",
        "Target BMI below 25 through structured diet plan",
        "150+ minutes of moderate exercise per week",
        "Monitor blood glucose daily",
        "Reduce refined carbohydrate intake"
    ],
    'Moderate Risk': [
        "Schedule annual diabetes screening",
        "Aim for BMI below 27",
        "30 minutes of daily physical activity",
        "Reduce sugary beverage consumption",
        "Monitor blood pressure regularly"
    ],
    'Low Risk': [
        "Maintain healthy weight and active lifestyle",
        "Annual wellness checkup",
        "Balanced diet with whole grains and vegetables",
        "Stay hydrated and limit processed foods"
    ]
}

print("=== Risk Tier Summary ===")
tier_counts = df_test['risk_tier'].value_counts()
for tier, count in tier_counts.items():
    pct = count / len(df_test) * 100
    print(f"  {tier:<15} {count:>4} patients ({pct:.1f}%)")

print("\n=== Lifestyle Interventions by Risk Tier ===")
for tier, tips in interventions.items():
    print(f"\n{tier}:")
    for tip in tips:
        print(f"  • {tip}")

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-23-diabetes-risk-predictor\outputs'
os.makedirs(output_dir, exist_ok=True)
df_test.to_csv(f'{output_dir}\\risk_scores.csv', index=False)
pd.DataFrame({
    'model':   ['XGBoost', 'Random Forest'],
    'roc_auc': [xgb_auc, rf_auc]
}).to_csv(f'{output_dir}\\model_results.csv', index=False)
print(f"\nExports saved to outputs/")

=== Risk Tier Summary ===
  Low Risk          83 patients (53.9%)
  High Risk         43 patients (27.9%)
  Moderate Risk     28 patients (18.2%)

=== Lifestyle Interventions by Risk Tier ===

High Risk:
  • Consult a physician immediately for glucose tolerance test
  • Target BMI below 25 through structured diet plan
  • 150+ minutes of moderate exercise per week
  • Monitor blood glucose daily
  • Reduce refined carbohydrate intake

Moderate Risk:
  • Schedule annual diabetes screening
  • Aim for BMI below 27
  • 30 minutes of daily physical activity
  • Reduce sugary beverage consumption
  • Monitor blood pressure regularly

Low Risk:
  • Maintain healthy weight and active lifestyle
  • Annual wellness checkup
  • Balanced diet with whole grains and vegetables
  • Stay hydrated and limit processed foods

Exports saved to outputs/
